In [4]:

import os
import sys
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)
# Set the parent directory as the current directory
os.chdir(parent_dir)

In [5]:
# load fully corrected datasets
from rdma.utils.data import read_json_file, print_json_structure


# all human labels
human_corrections_full = read_json_file("data/dataset/rare_disease_corrections_john.json")
print("-------- Human Corrections File ------")
print_json_structure(human_corrections_full)
# human + supervisor labels
human_rdma_corrections = read_json_file("data/dataset/adam_corrections_v2.json")
# supervisor labels
print("------- Supervisor Corrections File -------")
rdma_corrections = read_json_file("data/results/supervisor/multistage_no_min.json")
print_json_structure(rdma_corrections)

-------- Human Corrections File ------
Dictionary:
  metadata (dict): 
  Dictionary:
    timestamp (str): 
    total_entities_in_file (int): 
    reviewed_entities (int): 
  corrected_annotations (list): 
  List: (333 items)
    Item 0 (dict): 
    Dictionary:
      entity (str): 
      document_id (str): 
      orpha_code (str): 
      category (str): 
      is_rare_disease (bool): 
      ... and 2 more items
    Item 1 (dict): 
    Dictionary:
      entity (str): 
      document_id (str): 
      orpha_code (str): 
      category (str): 
      is_rare_disease (bool): 
      ... and 2 more items
    Item 2 (dict): 
    Dictionary:
      entity (str): 
      document_id (str): 
      orpha_code (str): 
      category (str): 
      is_rare_disease (bool): 
      ... and 2 more items
    Item 3 (dict): 
    Dictionary:
      entity (str): 
      document_id (str): 
      orpha_code (str): 
      category (str): 
      is_rare_disease (bool): 
      ... and 2 more items
    Item 4 (dict): 

# Comparing RDMA (only) and my annotations, because they go through the entire set of possible annotations

In [28]:
"""
Improved Rare Disease Annotator Agreement Analysis with Entity Clustering

This script computes the inter-annotator agreement between human corrections
and RDMA supervisor corrections for rare disease entity recognition,
using hierarchical clustering to group entity variants before comparison.
"""

import json
import numpy as np
from typing import Dict, List, Set, Tuple, Any, Optional
from sklearn.metrics import cohen_kappa_score
import scipy.stats as stats
from collections import defaultdict, Counter
from fuzzywuzzy import fuzz
import re
import networkx as nx
from tqdm import tqdm

def read_json_file(filename: str) -> dict:
    """Read a JSON file and return its contents."""
    with open(filename, 'r') as f:
        return json.load(f)

def normalize_entity(entity: str, abbreviations: Dict[str, str] = None) -> str:
    """
    Normalize entity text by lowercasing, removing extra spaces, 
    and expanding known abbreviations.
    
    Args:
        entity: The entity text to normalize
        abbreviations: Dictionary of abbreviations to expand
        
    Returns:
        Normalized entity text
    """
    if not entity:
        return ""
        
    # Convert to lowercase and strip
    normalized = entity.lower().strip()
    
    # Remove multiple spaces and replace with single space
    normalized = re.sub(r'\s+', ' ', normalized)
    
    # Remove hyphens between words (convert "heparin-induced" to "heparin induced")
    normalized = re.sub(r'(\w)-(\w)', r'\1 \2', normalized)
    
    # Expand abbreviation if it exists in the dictionary
    if abbreviations and normalized in abbreviations:
        return abbreviations[normalized]
        
    return normalized

def build_entity_similarity_graph(entities: List[str], threshold: int = 90) -> nx.Graph:
    """
    Build a graph where nodes are entities and edges exist if similarity exceeds threshold.
    
    Args:
        entities: List of entity strings
        threshold: Minimum similarity score to create an edge
        
    Returns:
        NetworkX graph with entities as nodes and similarities as edge weights
    """
    G = nx.Graph()
    
    # Add all entities as nodes
    for entity in entities:
        G.add_node(entity)
    
    # Add edges for similar entities
    n = len(entities)
    print(f"Building similarity graph for {n} entities...")
    
    # Use tqdm for progress tracking in the nested loop
    with tqdm(total=n*(n-1)//2) as pbar:
        for i in range(n):
            for j in range(i+1, n):
                # Update progress
                pbar.update(1)
                
                # Skip comparison if entities are identical
                entity1 = entities[i]
                entity2 = entities[j]
                if entity1 == entity2:
                    continue
                
                # Quick check for substring containment
                if entity1 in entity2 or entity2 in entity1:
                    # Calculate similarity
                    similarity = fuzz.ratio(entity1, entity2)
                    if similarity >= threshold:
                        G.add_edge(entity1, entity2, weight=similarity)
                    continue
                
                # For non-substring cases, use token sort ratio
                # Calculate token sort ratio for word order invariance
                token_similarity = fuzz.token_sort_ratio(entity1, entity2)
                if token_similarity >= threshold:
                    G.add_edge(entity1, entity2, weight=token_similarity)
    
    print(f"Graph built with {len(G.nodes)} nodes and {len(G.edges)} edges")
    return G

def cluster_entities(entities: List[str], threshold: int = 90) -> Dict[str, str]:
    """
    Cluster entities using connected components in a similarity graph.
    
    Args:
        entities: List of entity strings
        threshold: Minimum similarity score to consider entities similar
        
    Returns:
        Dictionary mapping each entity to its canonical form
    """
    # Build similarity graph
    G = build_entity_similarity_graph(entities, threshold)
    
    # Find connected components (clusters)
    clusters = list(nx.connected_components(G))
    print(f"Found {len(clusters)} entity clusters")
    
    # Create entity mapping
    entity_mapping = {}
    
    # Process each cluster
    for cluster in clusters:
        # Sort by length (prefer longer names as canonical)
        sorted_cluster = sorted(cluster, key=len, reverse=True)
        
        # Choose the longest entity as canonical form
        canonical = sorted_cluster[0]
        
        # Map all entities in this cluster to the canonical form
        for entity in cluster:
            entity_mapping[entity] = canonical
    
    # Add identity mappings for entities not in any cluster
    for entity in entities:
        if entity not in entity_mapping:
            entity_mapping[entity] = entity
    
    return entity_mapping

def extract_document_entity_sets_with_clustering(
    human_corrections: dict, 
    supervisor_corrections: dict, 
    similarity_threshold: int = 90
) -> Tuple[Dict[str, Dict[str, bool]], Dict[str, Dict[str, bool]], Dict[str, str]]:
    """
    Extract document-level entity sets from both annotations with entity clustering.
    
    Args:
        human_corrections: Human corrections dictionary
        supervisor_corrections: Supervisor corrections dictionary
        similarity_threshold: Minimum similarity to consider entities as the same
        
    Returns:
        Tuple of (human_doc_entities, supervisor_doc_entities, entity_mapping)
        Where entity_mapping maps variant spellings to canonical forms
    """
    # Dictionary for abbreviation expansion
    abbreviations = {
        "hit": "heparin induced thrombocytopenia",
        "pah": "pulmonary arterial hypertension",
        "pfo": "patent foramen ovale",
        "pcd": "primary ciliary dyskinesia",
        "hids": "hyper-igd syndrome",
        "ald": "adrenoleukodystrophy",
        # Add other abbreviations as needed
    }
    
    # Track excluded annotations
    excluded_annotations = {
        "bracketed_context": 0,
        "entity_not_in_context": 0,
        "excluded_entities": set()  # Set of excluded entity names
    }
    
    def is_valid_annotation(entity, context):
        """Check if an annotation is valid."""
        # Check if context is bracketed
        if context and context.strip().startswith('[') and context.strip().endswith(']'):
            excluded_annotations["bracketed_context"] += 1
            excluded_annotations["excluded_entities"].add(entity)
            return False
            
        # Check if entity exists in context
        if context and entity.lower() not in context.lower():
            # Check if it's an abbreviation that exists in expanded form
            expanded = False
            for abbr, expansion in abbreviations.items():
                if entity.lower() == abbr and expansion.lower() in context.lower():
                    expanded = True
                    break
                    
            if not expanded:
                excluded_annotations["entity_not_in_context"] += 1
                excluded_annotations["excluded_entities"].add(entity)
                return False
                
        # Filter out specific entities to exclude
        excluded_terms = ["high altitude pulmonary edema"]
        if any(term.lower() in entity.lower() for term in excluded_terms):
            excluded_annotations["excluded_entities"].add(entity)
            return False
            
        return True
    
    # Step 1: Collect all entities from both human and supervisor annotations
    all_entities = []
    raw_entity_to_original = {}  # Maps normalized entities to their original form
    raw_entity_to_docid = defaultdict(set)  # Maps normalized entities to document IDs
    
    # Human annotations
    if human_corrections and 'corrected_annotations' in human_corrections:
        for annotation in human_corrections['corrected_annotations']:
            if 'entity' in annotation and 'document_id' in annotation and 'is_rare_disease' in annotation:
                entity = annotation['entity']
                context = annotation.get('context', '')
                doc_id = annotation['document_id']
                
                # Skip invalid annotations
                if not is_valid_annotation(entity, context):
                    continue
                
                # Normalize entity
                normalized = normalize_entity(entity, abbreviations)
                all_entities.append(normalized)
                raw_entity_to_original[normalized] = entity
                raw_entity_to_docid[normalized].add(doc_id)
    
    # Supervisor annotations
    if supervisor_corrections and 'results' in supervisor_corrections:
        # Process all categories
        for category in ['true_positives', 'false_positives', 'false_negatives']:
            if category in supervisor_corrections['results']:
                for annotation in supervisor_corrections['results'][category]:
                    if 'entity' in annotation and 'document_id' in annotation and 'is_rare_disease' in annotation:
                        entity = annotation['entity']
                        context = annotation.get('context', '')
                        doc_id = annotation['document_id']
                        
                        # Skip invalid annotations
                        if not is_valid_annotation(entity, context):
                            continue
                        
                        # Normalize entity
                        normalized = normalize_entity(entity, abbreviations)
                        all_entities.append(normalized)
                        raw_entity_to_original[normalized] = entity
                        raw_entity_to_docid[normalized].add(doc_id)
    
    # Get unique entities for clustering
    unique_entities = list(set(all_entities))
    print(f"Found {len(unique_entities)} unique normalized entities across all annotations")
    
    # Step 2: Cluster entities and create mapping to canonical forms
    entity_mapping = cluster_entities(unique_entities, similarity_threshold)
    
    # Print cluster statistics
    canonical_forms = set(entity_mapping.values())
    print(f"Clustered {len(unique_entities)} entities into {len(canonical_forms)} canonical forms")
    
    # Create mapping from original form to canonical form
    original_to_canonical = {}
    for normalized, original in raw_entity_to_original.items():
        if normalized in entity_mapping:
            original_to_canonical[original] = entity_mapping[normalized]
    
    # Step 3: Initialize dictionaries to store document-level entities
    human_doc_entities = defaultdict(dict)  # {doc_id: {canonical_entity: is_rare_disease}}
    supervisor_doc_entities = defaultdict(dict)  # {doc_id: {canonical_entity: is_rare_disease}}
    
    # Step 4: Process human annotations with canonical forms
    if human_corrections and 'corrected_annotations' in human_corrections:
        for annotation in human_corrections['corrected_annotations']:
            if 'entity' in annotation and 'document_id' in annotation and 'is_rare_disease' in annotation:
                doc_id = annotation['document_id']
                entity = annotation['entity']
                context = annotation.get('context', '')
                is_rare = annotation['is_rare_disease']
                
                # Skip invalid annotations
                if not is_valid_annotation(entity, context):
                    continue
                
                # Get canonical form
                canonical_entity = original_to_canonical.get(entity, entity)
                
                # Add to document entities
                human_doc_entities[doc_id][canonical_entity] = is_rare
    
    # Step 5: Process supervisor annotations with canonical forms
    if supervisor_corrections and 'results' in supervisor_corrections:
        # Process all categories
        for category in ['true_positives', 'false_positives', 'false_negatives']:
            if category in supervisor_corrections['results']:
                for annotation in supervisor_corrections['results'][category]:
                    if 'entity' in annotation and 'document_id' in annotation and 'is_rare_disease' in annotation:
                        doc_id = annotation['document_id']
                        entity = annotation['entity']
                        context = annotation.get('context', '')
                        is_rare = annotation['is_rare_disease']
                        
                        # Skip invalid annotations
                        if not is_valid_annotation(entity, context):
                            continue
                        
                        # Get canonical form
                        canonical_entity = original_to_canonical.get(entity, entity)
                        
                        # Add to document entities
                        supervisor_doc_entities[doc_id][canonical_entity] = is_rare
    
    # Print summary of excluded annotations
    print(f"Excluded {excluded_annotations['bracketed_context']} annotations with bracketed context")
    print(f"Excluded {excluded_annotations['entity_not_in_context']} annotations where entity wasn't found in context")
    print(f"Total unique entities excluded: {len(excluded_annotations['excluded_entities'])}")
    
    # Get all documents from both sets to ensure we have complete coverage
    all_docs = set(human_doc_entities.keys()) | set(supervisor_doc_entities.keys())
    
    # Ensure each document is represented in both dictionaries, even if empty
    for doc_id in all_docs:
        if doc_id not in human_doc_entities:
            human_doc_entities[doc_id] = {}
        if doc_id not in supervisor_doc_entities:
            supervisor_doc_entities[doc_id] = {}
    
    # Print clustering examples
    print("\nExample entity clusters:")
    cluster_examples = {}
    for entity, canonical in entity_mapping.items():
        if entity != canonical:  # Only show actual mappings, not identity mappings
            if canonical not in cluster_examples:
                cluster_examples[canonical] = []
            cluster_examples[canonical].append(entity)
    
    # Show 10 most interesting clusters (those with most members)
    for canonical, variants in sorted(cluster_examples.items(), key=lambda x: len(x[1]), reverse=True)[:10]:
        print(f"  Canonical: '{canonical}'")
        print(f"  Variants: {', '.join([f'"{v}"' for v in variants[:5]])}")
        if len(variants) > 5:
            print(f"  ...and {len(variants)-5} more variants")
        print()
    
    return human_doc_entities, supervisor_doc_entities, entity_mapping

def compute_agreement_metrics(human_doc_entities: Dict[str, Dict[str, bool]], 
                             supervisor_doc_entities: Dict[str, Dict[str, bool]],
                             excluded_docs: Optional[List[str]] = None) -> Dict[str, Any]:
    """
    Compute agreement metrics between human and supervisor annotations without filtering by rarity.
    
    Args:
        human_doc_entities: Dictionary mapping document_id to a dict of {entity: is_rare_disease}
        supervisor_doc_entities: Dictionary mapping document_id to a dict of {entity: is_rare_disease}
        excluded_docs: Optional list of document IDs to exclude from evaluation
        
    Returns:
        Dictionary with agreement metrics
    """
    # Initialize counters
    all_judgments = []  # List of (human, supervisor) tuples for each entity judgment
    
    # Get all document IDs
    all_doc_ids = set(human_doc_entities.keys()) | set(supervisor_doc_entities.keys())
    
    # Remove excluded documents if specified
    if excluded_docs:
        all_doc_ids = all_doc_ids - set(excluded_docs)
    
    # Collect all unique entities across all documents
    all_unique_entities_set = set()
    all_doc_entity_pairs = []  # List of (doc_id, entity) pairs
    
    for doc_id in all_doc_ids:
        # Combine both sets of entities for this document
        human_entities = human_doc_entities[doc_id]
        supervisor_entities = supervisor_doc_entities[doc_id]
        all_entities = set(human_entities.keys()) | set(supervisor_entities.keys())
        
        for entity in all_entities:
            # Track unique entities
            all_unique_entities_set.add(entity)
            # Track all document-entity pairs
            all_doc_entity_pairs.append((doc_id, entity))
    
    # Lists for Cohen's Kappa and other metrics
    human_judgments = []
    supervisor_judgments = []
    
    # Count metrics for confusion matrix
    tp = 0  # Both say it's rare
    tn = 0  # Both say it's not rare
    fp = 0  # Supervisor says rare, Human says not rare
    fn = 0  # Human says rare, Supervisor says not rare
    
    # Process each document-entity pair
    for doc_id, entity in all_doc_entity_pairs:
        # Get human and supervisor judgments (default to False if entity is not present)
        human_judgment = human_doc_entities[doc_id].get(entity, False)
        supervisor_judgment = supervisor_doc_entities[doc_id].get(entity, False)
        
        # Add to lists for metrics calculation
        human_judgments.append(1 if human_judgment else 0)
        supervisor_judgments.append(1 if supervisor_judgment else 0)
        
        # Update confusion matrix
        if human_judgment and supervisor_judgment:
            tp += 1
        elif not human_judgment and not supervisor_judgment:
            tn += 1
        elif supervisor_judgment and not human_judgment:
            fp += 1
        elif human_judgment and not supervisor_judgment:
            fn += 1
        
        # Save the pair of judgments
        all_judgments.append((human_judgment, supervisor_judgment))
    
    # Calculate metrics
    total_judgments = len(all_judgments)
    agreements = tp + tn
    disagreements = fp + fn
    
    # Agreement rates
    if total_judgments > 0:
        percent_agreement = agreements / total_judgments
    else:
        percent_agreement = 0
    
    # Precision, recall, F1
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    # Calculate Cohen's Kappa
    if human_judgments and supervisor_judgments:
        kappa = cohen_kappa_score(human_judgments, supervisor_judgments)
    else:
        kappa = 0
    
    # Calculate Pearson correlation
    if human_judgments and supervisor_judgments and len(human_judgments) > 1:
        pearson_corr, p_value = stats.pearsonr(human_judgments, supervisor_judgments)
    else:
        pearson_corr = 0
        p_value = 1
    
    # Compile entity statistics (with rarity classifications)
    human_rare_entities = sum(1 for doc_id in all_doc_ids for entity, is_rare in human_doc_entities[doc_id].items() if is_rare)
    supervisor_rare_entities = sum(1 for doc_id in all_doc_ids for entity, is_rare in supervisor_doc_entities[doc_id].items() if is_rare)
    
    human_nonrare_entities = sum(1 for doc_id in all_doc_ids for entity, is_rare in human_doc_entities[doc_id].items() if not is_rare)
    supervisor_nonrare_entities = sum(1 for doc_id in all_doc_ids for entity, is_rare in supervisor_doc_entities[doc_id].items() if not is_rare)
    
    # Count unique entities
    unique_human_entities = set()
    unique_supervisor_entities = set()
    unique_human_rare = set()
    unique_supervisor_rare = set()
    
    for doc_id in all_doc_ids:
        for entity, is_rare in human_doc_entities[doc_id].items():
            unique_human_entities.add(entity)
            if is_rare:
                unique_human_rare.add(entity)
                
        for entity, is_rare in supervisor_doc_entities[doc_id].items():
            unique_supervisor_entities.add(entity)
            if is_rare:
                unique_supervisor_rare.add(entity)
    
    return {
        'total_documents': len(all_doc_ids),
        'total_entity_judgments': total_judgments,
        'total_agreements': agreements,
        'total_disagreements': disagreements,
        
        # Entity counts with rarity classification
        'human_rare_entities': human_rare_entities,
        'human_nonrare_entities': human_nonrare_entities,
        'supervisor_rare_entities': supervisor_rare_entities,
        'supervisor_nonrare_entities': supervisor_nonrare_entities,
        
        # Unique entity counts
        'unique_entities_total': len(all_unique_entities_set),
        'unique_human_entities': len(unique_human_entities),
        'unique_supervisor_entities': len(unique_supervisor_entities),
        'unique_human_rare': len(unique_human_rare),
        'unique_supervisor_rare': len(unique_supervisor_rare),
        
        # Confusion matrix
        'true_positives': tp,  # Both say it's rare
        'true_negatives': tn,  # Both say it's not rare
        'false_positives': fp,  # Supervisor says rare, Human says not rare  
        'false_negatives': fn,  # Human says rare, Supervisor says not rare
        
        # Agreement metrics
        'percent_agreement': percent_agreement,
        'cohen_kappa': kappa,
        'pearson_correlation': pearson_corr,
        'pearson_p_value': p_value,
        
        # Classification metrics
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

def analyze_disagreements(human_doc_entities: Dict[str, Dict[str, bool]], 
                         supervisor_doc_entities: Dict[str, Dict[str, bool]]) -> Dict[str, Any]:
    """
    Analyze disagreements in entity classification.
    """
    # Get all document IDs
    all_doc_ids = set(human_doc_entities.keys()) | set(supervisor_doc_entities.keys())
    
    # Initialize disagreement lists
    human_rare_supervisor_not = []  # Human says rare, Supervisor says not rare
    supervisor_rare_human_not = []  # Supervisor says rare, Human says not rare
    
    # To track contradictory classifications
    entity_classifications = defaultdict(lambda: {"human_rare": 0, "human_not_rare": 0, 
                                                 "supervisor_rare": 0, "supervisor_not_rare": 0,
                                                 "documents": set()})
    
    # Process each document
    for doc_id in all_doc_ids:
        human_entities = human_doc_entities.get(doc_id, {})
        supervisor_entities = supervisor_doc_entities.get(doc_id, {})
        
        # Get all entities from both sets
        all_entities = set(human_entities.keys()) | set(supervisor_entities.keys())
        
        for entity in all_entities:
            # Get judgments (default to False if not present)
            human_judgment = human_entities.get(entity, False)
            supervisor_judgment = supervisor_entities.get(entity, False)
            
            # Track all classifications for this entity
            entity_classifications[entity]["documents"].add(doc_id)
            if human_judgment:
                entity_classifications[entity]["human_rare"] += 1
            else:
                entity_classifications[entity]["human_not_rare"] += 1
            
            if supervisor_judgment:
                entity_classifications[entity]["supervisor_rare"] += 1
            else:
                entity_classifications[entity]["supervisor_not_rare"] += 1
            
            # Check for disagreements
            if human_judgment and not supervisor_judgment:
                # Human says rare, Supervisor says not rare
                human_rare_supervisor_not.append({
                    'entity': entity,
                    'document_id': doc_id
                })
            elif supervisor_judgment and not human_judgment:
                # Supervisor says rare, Human says not rare
                supervisor_rare_human_not.append({
                    'entity': entity,
                    'document_id': doc_id
                })
    
    # Count frequencies
    human_rare_freq = {}
    for item in human_rare_supervisor_not:
        entity = item['entity']
        if entity not in human_rare_freq:
            human_rare_freq[entity] = 0
        human_rare_freq[entity] += 1
    
    supervisor_rare_freq = {}
    for item in supervisor_rare_human_not:
        entity = item['entity']
        if entity not in supervisor_rare_freq:
            supervisor_rare_freq[entity] = 0
        supervisor_rare_freq[entity] += 1
    
    # Sort by frequency
    human_rare_sorted = sorted(human_rare_freq.items(), key=lambda x: x[1], reverse=True)
    supervisor_rare_sorted = sorted(supervisor_rare_freq.items(), key=lambda x: x[1], reverse=True)
    
    # Identify problematic entities with mixed classifications
    contradictory_entities = {}
    for entity, stats in entity_classifications.items():
        if stats["human_rare"] > 0 and stats["human_not_rare"] > 0:
            contradictory_entities[entity] = {
                "human_contradictory": True,
                "human_rare_count": stats["human_rare"],
                "human_not_rare_count": stats["human_not_rare"],
                "documents": list(stats["documents"])
            }
        if stats["supervisor_rare"] > 0 and stats["supervisor_not_rare"] > 0:
            if entity not in contradictory_entities:
                contradictory_entities[entity] = {
                    "human_contradictory": False,
                    "documents": list(stats["documents"])
                }
            contradictory_entities[entity]["supervisor_contradictory"] = True
            contradictory_entities[entity]["supervisor_rare_count"] = stats["supervisor_rare"]
            contradictory_entities[entity]["supervisor_not_rare_count"] = stats["supervisor_not_rare"]
    
    # For specific entities of interest
    entities_of_interest = ["heparin induced thrombocytopenia", "portal vein thrombosis", 
                          "tracheobronchomalacia", "rheumatic fever", "sarcoid"]
    entities_detail = {}
    
    # Find the closest match for each entity of interest
    for target_entity in entities_of_interest:
        best_match = None
        best_score = 0
        
        # Find the best matching entity in our classification data
        for entity in entity_classifications:
            # Try exact match first
            if entity.lower() == target_entity.lower():
                best_match = entity
                break
                
            # Otherwise use fuzzy matching
            score = fuzz.token_sort_ratio(entity.lower(), target_entity.lower())
            if score > best_score and score >= 85:  # At least 85% similarity
                best_score = score
                best_match = entity
        
        # If we found a match, get its details
        if best_match:
            stats = entity_classifications[best_match]
            entities_detail[target_entity] = {
                "matched_entity": best_match,
                "match_score": best_score if best_match.lower() != target_entity.lower() else 100,
                "stats": stats,
                "human_rare_documents": [doc_id for doc_id in stats["documents"] 
                                        if human_doc_entities.get(doc_id, {}).get(best_match, False)],
                "supervisor_rare_documents": [doc_id for doc_id in stats["documents"] 
                                            if supervisor_doc_entities.get(doc_id, {}).get(best_match, False)]
            }
    
    return {
        'human_rare_supervisor_not': human_rare_supervisor_not,
        'supervisor_rare_human_not': supervisor_rare_human_not,
        'human_rare_freq': human_rare_sorted,
        'supervisor_rare_freq': supervisor_rare_sorted,
        'total_human_rare_disagreements': len(human_rare_supervisor_not),
        'total_supervisor_rare_disagreements': len(supervisor_rare_human_not),
        'unique_human_rare_disagreements': len(human_rare_freq),
        'unique_supervisor_rare_disagreements': len(supervisor_rare_freq),
        'contradictory_entities': contradictory_entities,
        'entities_detail': entities_detail
    }

def print_report(metrics: Dict[str, Any], disagreements: Dict[str, Any]) -> None:
    """Print a report with the agreement metrics."""
    print("\n=== RARE DISEASE ANNOTATOR AGREEMENT REPORT WITH ENTITY CLUSTERING ===")
    
    print("\n--- ENTITY STATISTICS ---")
    print(f"Documents analyzed: {metrics['total_documents']}")
    print(f"Total entity judgments: {metrics['total_entity_judgments']}")
    
    print(f"\nHuman annotations:")
    print(f"  Total entities: {metrics['human_rare_entities'] + metrics['human_nonrare_entities']}")
    print(f"  Rare disease entities: {metrics['human_rare_entities']}")
    print(f"  Non-rare entities: {metrics['human_nonrare_entities']}")
    
    print(f"\nSupervisor annotations:")
    print(f"  Total entities: {metrics['supervisor_rare_entities'] + metrics['supervisor_nonrare_entities']}")
    print(f"  Rare disease entities: {metrics['supervisor_rare_entities']}")
    print(f"  Non-rare entities: {metrics['supervisor_nonrare_entities']}")
    
    print(f"\nUnique entities after clustering:")
    print(f"  Total unique entities: {metrics['unique_entities_total']}")
    print(f"  Unique in human annotations: {metrics['unique_human_entities']}")
    print(f"  Unique in supervisor annotations: {metrics['unique_supervisor_entities']}")
    print(f"  Unique rare in human: {metrics['unique_human_rare']}")
    print(f"  Unique rare in supervisor: {metrics['unique_supervisor_rare']}")
    
    print("\n--- AGREEMENT METRICS ---")
    print(f"Agreements: {metrics['total_agreements']} of {metrics['total_entity_judgments']} judgments")
    print(f"Percentage Agreement (Accuracy): {metrics['percent_agreement']:.4f}")
    print(f"Cohen's Kappa: {metrics['cohen_kappa']:.4f}")
    
    print("\n--- CONFUSION MATRIX ---")
    print(f"True Positives (both rare): {metrics['true_positives']}")
    print(f"True Negatives (both non-rare): {metrics['true_negatives']}")
    print(f"False Positives (supervisor rare, human non-rare): {metrics['false_positives']}")
    print(f"False Negatives (human rare, supervisor non-rare): {metrics['false_negatives']}")
    
    print("\n--- CLASSIFICATION METRICS ---")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"F1 Score: {metrics['f1']:.4f}")
    
    print("\n--- DISAGREEMENT SUMMARY ---")
    print(f"Human says rare, supervisor says not: {disagreements['total_human_rare_disagreements']} disagreements ({disagreements['unique_human_rare_disagreements']} unique entities)")
    print(f"Supervisor says rare, human says not: {disagreements['total_supervisor_rare_disagreements']} disagreements ({disagreements['unique_supervisor_rare_disagreements']} unique entities)")
    
    if disagreements['human_rare_freq']:
        print("\nTop disagreements (human says rare, supervisor says not):")
        for entity, count in disagreements['human_rare_freq'][:5]:
            print(f"  {entity}: {count} occurrences")
    
    if disagreements['supervisor_rare_freq']:
        print("\nTop disagreements (supervisor says rare, human says not):")
        for entity, count in disagreements['supervisor_rare_freq'][:5]:
            print(f"  {entity}: {count} occurrences")
            
    # Print contradictory entities
    if disagreements.get('contradictory_entities'):
        print("\nEntities with contradictory classifications:")
        for entity, details in sorted(disagreements['contradictory_entities'].items(), 
                                    key=lambda x: x[1].get('human_rare_count', 0) + 
                                                 x[1].get('supervisor_rare_count', 0), 
                                    reverse=True)[:10]:
            print(f"\n  {entity}:")
            if details.get('human_contradictory', False):
                print(f"    Human classified as rare: {details['human_rare_count']} times")
                print(f"    Human classified as not rare: {details['human_not_rare_count']} times")
            if details.get('supervisor_contradictory', False):
                print(f"    Supervisor classified as rare: {details['supervisor_rare_count']} times")
                print(f"    Supervisor classified as not rare: {details['supervisor_not_rare_count']} times")
            print(f"    Appears in {len(details['documents'])} documents")
    
    # Print details for entities of interest
    if disagreements.get('entities_detail'):
        print("\nSpecific entities of interest:")
        for target_entity, details in disagreements['entities_detail'].items():
            matched_entity = details.get("matched_entity", "")
            match_info = ""
            if matched_entity != target_entity and details.get("match_score", 0) < 100:
                match_info = f" (matched to '{matched_entity}' with {details['match_score']}% similarity)"
                
            stats = details['stats']
            print(f"\n  {target_entity}{match_info}:")
            print(f"    Human classified as rare: {stats['human_rare']} times")
            print(f"    Human classified as not rare: {stats['human_not_rare']} times")
            print(f"    Supervisor classified as rare: {stats['supervisor_rare']} times")
            print(f"    Supervisor classified as not rare: {stats['supervisor_not_rare']} times")
            print(f"    Human rare documents: {details['human_rare_documents']}")
            print(f"    Supervisor rare documents: {details['supervisor_rare_documents']}")

# Example usage:
if __name__ == "__main__":
    # Load human and supervisor corrections
    human_corrections = read_json_file("data/dataset/rare_disease_corrections_john.json")
    rdma_corrections = read_json_file("data/results/supervisor/multistage_no_min.json")

    # Extract document-level entity sets with entity clustering
    human_doc_entities, supervisor_doc_entities, entity_mapping = extract_document_entity_sets_with_clustering(
        human_corrections, 
        rdma_corrections,
        similarity_threshold=90  # Using 90% as the clustering threshold
    )
    
    # Compute agreement metrics
    metrics = compute_agreement_metrics(
        human_doc_entities, 
        supervisor_doc_entities
    )
    
    # Analyze disagreements
    disagreements = analyze_disagreements(
        human_doc_entities, 
        supervisor_doc_entities
    )
    
    # Print report
    print_report(metrics, disagreements)

SyntaxError: f-string: unterminated string (2512466829.py, line 344)

In [9]:
print(human_doc_entities)
print(supervisor_doc_entities)

defaultdict(<class 'set'>, {'1208': {'nocardiosis'}, '950': {'retinitis pigmentosa'}, '977': {'rheumatic fever'}, '1552': {'hit'}, '1790': {'tracheobronchomalacia'}, '2452': {'sarcoidosis'}, '3390': {'als'}, '4806': {'rheumatic fever'}, '4938': {'sarcoid'}, '10406': {'medullary sponge kidney'}, '6465': {'hit'}, '13231': {'hyperthyroidism'}, '7688': {'cervical stenosis', 'amyotrophic lateral sclerosis'}, '6188': {'pml', 'hyperthyroidism'}, '8979': {'heparin induced thrombocytopenia', 'sclerosis cholangitis', 'hit'}, '10004': {'asbestosis', 'dilated cardiomyopathy'}, '10715': {'heparin induced thrombocytopenia'}, '9512': {'mediastinitis'}, '8960': {'sick sinus syndrome', 'sarcoid', "bechet's disease"}, '16334': {'cervical stenosis', 'hit'}, '16347': {'hypothyroidism secondary'}, '13666': {'central nervous system and systemic lymphoma'}, '14936': {'bullous pemphigoid', 'methemoglobinemia'}, '11938': {'retinopathy of prematurity'}, '11604': {'antiphospholipid antibody syndrome', 'microcyti

In [12]:
# print(sorted(supervisor_doc_entities.keys()))
for doc_id, entities in human_doc_entities.items():
    print(f"Document ID: {doc_id}")
    print(f"Entities: {entities}")
    print("Supervisor Entities:")
    if doc_id in supervisor_doc_entities:
        print(supervisor_doc_entities[doc_id])
    else:
        print("No supervisor entities found for this document.")
    print()
# print(sorted(human_doc_entities.keys()))

Document ID: 1208
Entities: {'nocardiosis'}
Supervisor Entities:
set()

Document ID: 950
Entities: {'retinitis pigmentosa'}
Supervisor Entities:
{'retinitis pigmentosa'}

Document ID: 977
Entities: {'rheumatic fever'}
Supervisor Entities:
set()

Document ID: 1552
Entities: {'hit'}
Supervisor Entities:
{'heparin-induced thrombocytopenia'}

Document ID: 1790
Entities: {'tracheobronchomalacia'}
Supervisor Entities:
set()

Document ID: 2452
Entities: {'sarcoidosis'}
Supervisor Entities:
{'sarcoidosis'}

Document ID: 3390
Entities: {'als'}
Supervisor Entities:
{'amyotrophic lateral sclerosis'}

Document ID: 4806
Entities: {'rheumatic fever'}
Supervisor Entities:
set()

Document ID: 4938
Entities: {'sarcoid'}
Supervisor Entities:
set()

Document ID: 10406
Entities: {'medullary sponge kidney'}
Supervisor Entities:
{'hit'}

Document ID: 6465
Entities: {'hit'}
Supervisor Entities:
{'hit'}

Document ID: 13231
Entities: {'hyperthyroidism'}
Supervisor Entities:
set()

Document ID: 7688
Entities: {

# All Human vs. RDMA + Human

# 